# Performance and Optimization

"*Premature optimization is the root of all evil.*" - Donald Knuth

Before making code faster, you must first make it work correctly. Once it works, if it's too slow, you need to **measure** it to find the actual bottlenecks before changing anything. 

This notebook covers how to measure execution time, profile your code to find bottlenecks, and use built-in Python features to drastically improve performance.

## 1. Timing Code Execution

The simplest way to check performance is to time it. While you can use the `time` module, Python provides the `timeit` module for accurate micro-benchmarking.

In Jupyter Notebooks, we have access to a special "magic command" called `%timeit`.

In [1]:
# %timeit runs the single line of code multiple times to get a statistically accurate average
print("Timing a list comprehension:")
%timeit [x**2 for x in range(10_000)]

print("\nTiming the map() function:")
# map() is often faster than list comprehensions because it's implemented in C
%timeit list(map(lambda x: x**2, range(10_000)))

print("\nTiming generator expression (lazy evaluation):")
# Generators don't compute all values at once, so creation is nearly instantaneous
%timeit (x**2 for x in range(10_000))

Timing a list comprehension:
352 μs ± 1.54 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

Timing the map() function:
527 μs ± 4.59 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

Timing generator expression (lazy evaluation):
127 ns ± 0.689 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


## 2. Profiling Code with `cProfile`

Timing tells you *how long* a script takes. Profiling tells you *where* the time is being spent. The built-in `cProfile` module breaks down execution time function by function.

In [2]:
import cProfile
import time

def slow_function():
    time.sleep(0.5)  # Simulating a slow I/O task (like a database query)
    return "Done"

def heavy_computation():
    return sum([x**2 for x in range(2_000_000)])

def main_app():
    slow_function()
    heavy_computation()

# Run the profiler on our main app
# Look at 'tottime' (total time in function) and 'cumtime' (time in function + sub-functions)
cProfile.run('main_app()')

         687 function calls (681 primitive calls) in 0.629 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.433    0.433 1743565775.py:11(main_app)
        1    0.000    0.000    0.313    0.313 1743565775.py:4(slow_function)
        1    0.106    0.106    0.120    0.120 1743565775.py:8(heavy_computation)
        2    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        4    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1401(_handle_fromlist)
      2/1    0.000    0.000    0.629    0.629 <string>:1(<module>)
        1    0.000    0.000    0.000    0.000 asyncio.py:206(_handle_events)
        1    0.000    0.000    0.000    0.000 asyncio.py:231(add_callback)
        4    0.000    0.000    0.000    0.000 attrsettr.py:43(__getattr__)
        4    0.000    0.000    0.000    0.000 attrsettr.py:66(_get_attr_opt)
        1    0.000    0.000    0.000    0.000 b

## 3. Data Structures: Lists vs. Sets

Choosing the right data structure is the most impactful optimization you can make. 

- **Lists**: Great for ordered data and iteration. Terrible for lookups (checking if an item exists takes $O(N)$ time).
- **Sets**: Unordered, but use hash tables. Lookups take $O(1)$ time, making them exponentially faster for large datasets.

In [3]:
import time

# Setup a large dataset
large_list = list(range(10_000_000))
large_set = set(large_list)

target_number = 9_999_999  # Worst-case scenario (end of the list)

# --- List Lookup ---
start_time = time.perf_counter()
found_in_list = target_number in large_list
list_time = time.perf_counter() - start_time

# --- Set Lookup ---
start_time = time.perf_counter()
found_in_set = target_number in large_set
set_time = time.perf_counter() - start_time

print(f"List Lookup Time: {list_time:.6f} seconds")
print(f"Set Lookup Time:  {set_time:.6f} seconds")

if set_time > 0:
    print(f"\nThe Set was {list_time / set_time:,.0f}x faster!")

List Lookup Time: 0.063744 seconds
Set Lookup Time:  0.000024 seconds

The Set was 2,698x faster!


## 4. String Concatenation

Strings in Python are immutable. Every time you use `+` to add to a string, Python creates a brand new string in memory. If you are building a large string in a loop, you should use `.join()` instead.

In [4]:
words = ["Hello", "world", "this", "is", "a", "test"] * 1000

def slow_concatenation():
    result = ""
    for word in words:
        result += word + " "  # Creates a new string every loop iteration
    return result

def fast_concatenation():
    # .join() calculates total memory needed once, then builds the string
    return " ".join(words)

print("Timing += operator:")
%timeit slow_concatenation()

print("\nTiming .join():")
%timeit fast_concatenation()

Timing += operator:
338 μs ± 418 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

Timing .join():
42.6 μs ± 283 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 5. Caching and Memoization (`functools.lru_cache`)

If you have a function that performs expensive calculations and is often called with the same arguments, you can cache its results. Python's `functools` module makes this trivial.

In [5]:
from functools import lru_cache

# A notoriously slow recursive function
def fibonacci_slow(n):
    if n < 2:
        return n
    return fibonacci_slow(n-1) + fibonacci_slow(n-2)

# The exact same function, but we add the LRU (Least Recently Used) cache decorator
# maxsize determines how many recent calls to remember
@lru_cache(maxsize=128)
def fibonacci_fast(n):
    if n < 2:
        return n
    return fibonacci_fast(n-1) + fibonacci_fast(n-2)

print("Calculating fibonacci(35) without cache...")
%timeit -n 1 -r 1 fibonacci_slow(35)  # Runs only once because it's slow

print("\nCalculating fibonacci(35) WITH cache...")
%timeit fibonacci_fast(35)

print("\nCache stats:", fibonacci_fast.cache_info())

Calculating fibonacci(35) without cache...
950 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)

Calculating fibonacci(35) WITH cache...
35.7 ns ± 0.871 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)

Cache stats: CacheInfo(hits=81111143, misses=36, maxsize=128, currsize=36)
